In [0]:
import requests
import pandas as pd
from time import sleep

#puxar todos os dados
BASE_URL = "https://api.openbrewerydb.org/v1/breweries"
PER_PAGE = 200  #usar o maximo por pagina pela config da API
TIMEOUT = 2 
SLEEP_BETWEEN_CALLS = 0.2 

def fetch_all_breweries(max_pages=None, verbose=True):
    session = requests.Session()
    all_pages = []
    page = 1

    try:
        while True:
            params = {"page": page, "per_page": PER_PAGE}
            if verbose:
                print(f"Buscando página {page}...")

            resp = session.get(BASE_URL, params=params, timeout=TIMEOUT)
            resp.raise_for_status()  # lança exceção para códigos 4xx/5xx

            data = resp.json()
            if not isinstance(data, list):
                raise ValueError("Resposta inesperada da API (não é lista).")

            if not data:  # lista vazia → acabou a paginação
                if verbose:
                    print("Sem mais resultados. Paginação encerrada.")
                break

            # Armazena bloco desta página
            all_pages.append(pd.DataFrame(data))


            if max_pages is not None and page >= max_pages:
                if verbose:
                    print(f"Limite de {max_pages} páginas atingido.")
                break

            page += 1
            sleep(SLEEP_BETWEEN_CALLS)

    except requests.HTTPError as e:
        print(f"Erro HTTP na página {page}: {e} | Conteúdo: {getattr(resp, 'text', '')[:200]}")
    except requests.RequestException as e:
        print(f"Erro de rede/timeout na página {page}: {e}")
    except ValueError as e:
        print(f"Erro de parsing: {e}")

    # Concatena tudo 
    if all_pages:
        df = pd.concat(all_pages, ignore_index=True)
    else:
        df = pd.DataFrame()

    return df



#pegar todas as paginas
if __name__ == "__main__":
    df_breweries = fetch_all_breweries(max_pages=None, verbose=True)

    print(f"Total de registros: {len(df_breweries)}")
    print(df_breweries.head(10))

In [0]:
# Adicionando coluna com data da ingestão
from datetime import datetime

df_breweries_data = df_breweries.copy()
df_breweries_data["data_ingestão"] = datetime.now()
display(df_breweries_data)

In [0]:
#Salvando em parquet para lake de dados
df_breweries_data.to_parquet("raw_output/breweries.parquet")

In [0]:
%sql
--Opicional para databricks
CREATE DATABASE IF NOT EXISTS dbt_beer;
USE SCHEMA dbt_beer


In [0]:
#subi tabela para DW relacional (opicial "SQL-king")
df_breweries_spark = spark.createDataFrame(df_breweries_data)

(df_breweries_spark.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable("dbt_beer.breweries_raw"))

user_raw_breweries = spark.table("dbt_beer.breweries_raw")
display(user_raw_breweries)